In [ ]:
%%capture
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
%%capture
!pip uninstall mamba-ssm causal-conv1d torch torchvision torchaudio -y
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip cache purge
MAX_JOBS=1
!pip install causal-conv1d>=1.4.0 --no-cache-dir --no-build-isolation --no-binary :all:
MAX_JOBS=1
!pip install mamba-ssm --no-cache-dir --no-build-isolation --no-binary :all:


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import re
import math
import random
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Dict
from collections import Counter
from copy import deepcopy

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import torchvision.transforms as transforms

from mamba_ssm import Mamba

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
print(f'Device: {device}')
if torch.cuda.is_available():
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory // 1024**3} GB VRAM')


In [ ]:
DATA_ROOTS = [
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_clean',
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_20',
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_15',
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_10',
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_5',
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_0',
]

CLASS_NAMES  = sorted(['ardrone', 'background', 'bepop', 'phantom'])
NUM_CLASSES  = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

print('Class registry:')
for idx, name in IDX_TO_CLASS.items():
    print(f'  {idx}: {name}')


@dataclass
class Config:
    img_size:         int   = 224
    patch_size:       int   = 16
    in_channels:      int   = 3
    num_classes:      int   = NUM_CLASSES

    embed_dim:        int   = 192
    depth:            int   = 8
    # FIX: 16->32. Harmonic-interval memory requires more SSM state capacity.
    # ardrone vs bepop differ in frequency band spacing (harmonics ~5-10 patches apart).
    # d_state=16 was too small to hold that context across the sequence.
    d_state:          int   = 32
    d_conv:           int   = 4
    expand:           int   = 2
    drop_rate:        float = 0.10
    # FIX: 0.15->0.10. Small dataset (3.5K) — less stochastic depth needed.
    drop_path_rate:   float = 0.10
    # FIX: 0.1->1.0. layer_scale 0.1 was throttling SSM gradients by 10x
    # from epoch 1, slowing convergence significantly on a small dataset.
    layer_scale_init: float = 1.0

    batch_size:       int   = 32
    learning_rate:    float = 1e-4   # AdamW only — SGD removed (worse by >36%)
    weight_decay:     float = 1e-3

    epochs:           int   = 150
    # FIX: 30->40. More room since model is now learning properly.
    patience:         int   = 40
    grad_clip_norm:   float = 1.0
    # FIX: 0.1->0.05. Heavy smoothing hurts small-dataset class boundaries.
    label_smoothing:  float = 0.05
    # FIX: 0.3->0.2. Slightly gentler Mixup — was over-blending discriminative patches.
    mixup_alpha:      float = 0.2
    # FIX: 0.3->0.0. CutMix disabled. Stacked with SpecAugment it obliterates
    # the frequency-band patterns that separate ardrone from bepop.
    cutmix_alpha:     float = 0.0
    warmup_epochs:    int   = 10
    val_split:        float = 0.20
    random_state:     int   = 42


config = Config()
H_p = W_p = config.img_size // config.patch_size
print(f'\nConfig loaded')
print(f'  patch_size={config.patch_size}  grid={H_p}x{W_p}  tokens={H_p*W_p}')
print(f'  embed_dim={config.embed_dim}  depth={config.depth}  d_state={config.d_state}')
print(f'  adamw_lr={config.learning_rate}')
print(f'  label_smoothing={config.label_smoothing}  mixup_alpha={config.mixup_alpha}  cutmix_alpha={config.cutmix_alpha}')
print(f'  layer_scale_init={config.layer_scale_init}  grad_clip_norm={config.grad_clip_norm}')
print(f'  epochs={config.epochs}  warmup={config.warmup_epochs}  patience={config.patience}')
print(f'  batch_size={config.batch_size}')


In [ ]:
def parse_snr_folder(folder_name: str) -> int:
    if 'clean' in folder_name.lower():
        return 100
    match = re.search(r'snr_?(-?\d+)', folder_name)
    return int(match.group(1)) if match else 0


def load_dronerf_dataset(data_roots: List[str]) -> Tuple[List[str], np.ndarray, np.ndarray]:
    paths, labels, snrs = [], [], []
    print('Loading dataset...')
    class_counts = Counter()
    for snr_folder in data_roots:
        snr_val = parse_snr_folder(Path(snr_folder).name)
        if not Path(snr_folder).exists():
            print(f'  WARNING: Missing folder: {snr_folder}')
            continue
        for class_name in CLASS_NAMES:
            class_dir = Path(snr_folder) / class_name
            if not class_dir.exists():
                continue
            png_files = sorted(class_dir.glob('*.png'))
            lbl = CLASS_TO_IDX[class_name]
            for p in png_files:
                paths.append(str(p))
                labels.append(lbl)
                snrs.append(snr_val)
                class_counts[class_name] += 1
    print(f'\nTotal images: {len(paths)}')
    for c, n in sorted(class_counts.items()):
        print(f'  {c:<12}: {n:5d} images')
    return paths, np.array(labels, dtype=np.int64), np.array(snrs, dtype=np.int32)


all_paths, all_labels, all_snrs = load_dronerf_dataset(DATA_ROOTS)

indices = np.arange(len(all_paths))
train_idx, val_idx = train_test_split(
    indices,
    test_size=config.val_split,
    random_state=config.random_state,
    stratify=all_labels,
)
train_idx = sorted(train_idx.tolist())
val_idx   = sorted(val_idx.tolist())

train_paths  = [all_paths[i]  for i in train_idx]
train_labels = all_labels[train_idx]
train_snrs   = all_snrs[train_idx]

val_paths    = [all_paths[i]  for i in val_idx]
val_labels   = all_labels[val_idx]
val_snrs     = all_snrs[val_idx]

print(f'\nTrain: {len(train_paths)} | Val: {len(val_paths)}')
print(f'Train class counts: {dict(Counter(train_labels.tolist()))}')
print(f'Val   class counts: {dict(Counter(val_labels.tolist()))}')


In [ ]:
import hashlib
os.makedirs('vim_dronerf_outputs', exist_ok=True)

print('=' * 60)
print('DATASET SANITY CHECKS')
print('=' * 60)

# 1. Class balance per SNR level
print('\n[1] Class distribution at each SNR level (training set):')
print(f'  {"SNR":<10}', '  '.join(f'{c[:6]:>7}' for c in CLASS_NAMES))
for snr_val in sorted(set(train_snrs.tolist())):
    mask   = train_snrs == snr_val
    counts = Counter(train_labels[mask].tolist())
    label  = 'clean' if snr_val == 100 else f'{snr_val} dB'
    row    = '  '.join(f'{counts.get(i, 0):>7d}' for i in range(NUM_CLASSES))
    print(f'  {label:<10}  {row}')

# 2. Image integrity check
print('\n[2] Image integrity check (sampling 200 random paths)...')
rng     = np.random.default_rng(0)
sample  = rng.choice(len(all_paths), min(200, len(all_paths)), replace=False)
corrupt = []
for i in sample:
    try:
        img = Image.open(all_paths[i]).convert('RGB')
        arr = np.array(img)
        if arr.std() < 1e-3:
            corrupt.append((all_paths[i], 'blank image'))
    except Exception as e:
        corrupt.append((all_paths[i], str(e)))
if corrupt:
    print(f'  WARNING: {len(corrupt)} corrupt/blank images found:')
    for p, reason in corrupt[:5]:
        print(f'    {p}  ->  {reason}')
else:
    print('  All sampled images are valid and non-blank. OK')

# 3. Per-class pixel statistics
print('\n[3] Per-class pixel mean (should differ between drone types):')
_to_t = transforms.ToTensor()
class_pixels = {i: [] for i in range(NUM_CLASSES)}
_rng2 = np.random.default_rng(1)
for cls_idx in range(NUM_CLASSES):
    cls_mask = (all_labels == cls_idx)
    cls_paths_c = [all_paths[i] for i in range(len(all_paths)) if cls_mask[i]]
    sample_cls = _rng2.choice(len(cls_paths_c), min(80, len(cls_paths_c)), replace=False)
    for si in sample_cls:
        img = _to_t(Image.open(cls_paths_c[si]).convert('RGB'))
        class_pixels[cls_idx].append(img.mean().item())
for i, name in IDX_TO_CLASS.items():
    vals = class_pixels[i]
    print(f'  {name:<12}  mean_pixel={np.mean(vals):.4f}  std={np.std(vals):.4f}  n={len(vals)}')

# 4. Duplicate detection
print('\n[4] Duplicate check (hashing 300 random images)...')
_rng3    = np.random.default_rng(2)
_sample3 = _rng3.choice(len(all_paths), min(300, len(all_paths)), replace=False)
_hashes  = {}
dupes    = 0
for i in _sample3:
    with open(all_paths[i], 'rb') as f:
        h = hashlib.md5(f.read()).hexdigest()
    if h in _hashes:
        dupes += 1
    else:
        _hashes[h] = all_paths[i]
print(f'  Duplicates found in sample: {dupes}  ({"WARNING" if dupes > 5 else "OK"})')

# 5. Visualise 3 samples per class
print('\n[5] Saving sample images to vim_dronerf_outputs/sanity_samples.png ...')
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(9, 3 * NUM_CLASSES))
for row, cls_idx in enumerate(range(NUM_CLASSES)):
    cls_mask = (all_labels == cls_idx)
    cls_paths_v = [all_paths[i] for i in range(len(all_paths)) if cls_mask[i]]
    picks = np.random.default_rng(3).choice(len(cls_paths_v), 3, replace=False)
    for col, pi in enumerate(picks):
        img = Image.open(cls_paths_v[pi]).convert('RGB')
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_title(IDX_TO_CLASS[cls_idx], fontsize=9)
plt.tight_layout()
plt.savefig('vim_dronerf_outputs/sanity_samples.png', dpi=80, bbox_inches='tight')
plt.close()
print('  Saved sanity_samples.png')

# 6. Train/val overlap
print('\n[6] Train/val path overlap check...')
overlap = set(train_paths) & set(val_paths)
print(f'  Overlapping paths: {len(overlap)}  ({"WARNING" if overlap else "No overlap OK"})')

print('\n' + '=' * 60)
print('Sanity checks complete.')
print('=' * 60)


In [ ]:
def compute_spectrogram_stats(paths: List[str], sample_size: int = 2000) -> Tuple[List[float], List[float]]:
    rng  = np.random.default_rng(42)
    idx  = rng.choice(len(paths), min(sample_size, len(paths)), replace=False)
    to_t = transforms.ToTensor()
    pixel_lists = [[] for _ in range(3)]
    for i in idx:
        img = to_t(Image.open(paths[i]).convert('RGB'))
        for c in range(3):
            pixel_lists[c].append(img[c].reshape(-1))
    mean, std = [], []
    for c in range(3):
        all_pixels = torch.cat(pixel_lists[c])
        mean.append(all_pixels.mean().item())
        std.append(all_pixels.std().item())
    return mean, std


print('Computing global normalisation stats from training set...')
SPEC_MEAN, SPEC_STD = compute_spectrogram_stats(train_paths, sample_size=2000)
print(f'  SPEC_MEAN = {[round(v, 4) for v in SPEC_MEAN]}')
print(f'  SPEC_STD  = {[round(v, 4) for v in SPEC_STD]}')


In [ ]:
class SpecAugment:
    """Time-frequency masking applied AFTER normalisation."""
    def __init__(self, freq_mask_param=16, time_mask_param=16,
                 num_freq_masks=1, num_time_masks=1):
        self.F  = freq_mask_param
        self.T  = time_mask_param
        self.nF = num_freq_masks
        self.nT = num_time_masks

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        _, H, W = x.shape
        for _ in range(self.nF):
            f  = random.randint(0, self.F)
            f0 = random.randint(0, max(H - f, 0))
            x[:, f0:f0 + f, :] = 0.0
        for _ in range(self.nT):
            t  = random.randint(0, self.T)
            t0 = random.randint(0, max(W - t, 0))
            x[:, :, t0:t0 + t] = 0.0
        return x


class FreqShift:
    """
    Circular shift along the frequency axis (vertical).
    WHY: Forces the model to learn relative frequency patterns (harmonic
    spacing) rather than memorising absolute band positions.
    FIX: Reduced max_shift 28->14 (was erasing too much discriminative structure).
    """
    def __init__(self, max_shift: int = 14):
        self.max_shift = max_shift

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        shift = random.randint(-self.max_shift, self.max_shift)
        return torch.roll(x, shift, dims=1)


def get_train_transforms(img_size: int = 224) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),    # time-flip: OK, signal is time-symmetric
        # FIX: RandomVerticalFlip REMOVED.
        # Flipping vertically inverts the frequency axis — an ardrone signal at its
        # characteristic bands gets mapped to bepop's band positions.
        # This was silently mislabelling ~20% of training images.
        transforms.ColorJitter(brightness=0.15, contrast=0.15),  # gentler than 0.25
        transforms.ToTensor(),
        transforms.Normalize(mean=SPEC_MEAN, std=SPEC_STD),
        # FIX: 2 masks x 24px -> 1 mask x 16px.
        # Previously could blank up to 21% of the image height/width per mask.
        # Stacked with Mixup this was destroying freq-band patterns before the model saw them.
        SpecAugment(freq_mask_param=16, time_mask_param=16,
                    num_freq_masks=1, num_time_masks=1),
        FreqShift(max_shift=14),
    ])


def get_val_transforms(img_size: int = 224) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=SPEC_MEAN, std=SPEC_STD),
    ])


class DroneRFDataset(Dataset):
    def __init__(self, paths, labels, snrs, transform=None):
        self.paths     = paths
        self.labels    = torch.tensor(labels, dtype=torch.long)
        self.snrs      = snrs
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


def make_balanced_sampler(labels: np.ndarray):
    from torch.utils.data import WeightedRandomSampler
    counts  = Counter(labels.tolist())
    weights = torch.tensor(
        [1.0 / counts[int(l)] for l in labels], dtype=torch.float64)
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


def build_dataloaders(cfg, tr_paths, tr_labels, tr_snrs,
                      vl_paths, vl_labels, vl_snrs, num_workers=2):
    train_ds = DroneRFDataset(tr_paths, tr_labels, tr_snrs,
                              transform=get_train_transforms(cfg.img_size))
    val_ds   = DroneRFDataset(vl_paths, vl_labels, vl_snrs,
                              transform=get_val_transforms(cfg.img_size))
    sampler  = make_balanced_sampler(tr_labels)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                              sampler=sampler, num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader


_tl, _vl = build_dataloaders(
    config, train_paths, train_labels, train_snrs,
    val_paths, val_labels, val_snrs)
imgs, lbls = next(iter(_tl))
print(f'Train batches: {len(_tl)} | Val batches: {len(_vl)}')
print(f'Batch image shape: {imgs.shape}  label shape: {lbls.shape}')
print(f'Image range: [{imgs.min():.3f}, {imgs.max():.3f}]')
print(f'Batch class distribution (~8 each): {dict(Counter(lbls.tolist()))}')
del _tl, _vl, imgs, lbls


In [ ]:
class StochasticDepth(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        rand  = torch.rand(shape, dtype=x.dtype, device=x.device)
        rand  = torch.floor(rand + keep_prob)
        return (x / keep_prob) * rand


class LayerScale(nn.Module):
    def __init__(self, dim, init_value=1.0):
        super().__init__()
        self.gamma = nn.Parameter(init_value * torch.ones(dim))

    def forward(self, x):
        return x * self.gamma


class PatchEmbed(nn.Module):
    def __init__(self, img_size, patch_size, in_chans, embed_dim):
        super().__init__()
        self.H_p = img_size // patch_size
        self.W_p = img_size // patch_size
        self.num_patches = self.H_p * self.W_p
        self.proj = nn.Conv2d(in_chans, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)  # (B, N, D)


class VimBlock2D(nn.Module):
    """
    4-Directional Vision Mamba Block with learned direction merge.

    KEY FIX — concat+project instead of sum:
    ────────────────────────────────────────
    The old merge was: out = layer_scale(y_hf + y_hb + y_vf + y_vb)
    This is lossy. H and V scans encode structurally different information:
      H-scans: time evolution (how a frequency band changes over time)
      V-scans: frequency profiles (which bands activate together)
    When you sum them in the same D-dimensional space, the dominant scan
    direction overwrites the weaker one. For ardrone vs bepop, the V-scan
    (frequency profile) is the key discriminator — but the H-scan can drown it.

    FIX: Concatenate all 4 outputs -> (B, N, D*4), then project back to D.
    The linear projection learns WHICH directions matter for EACH token position.
    Tokens at discriminative frequency bands will up-weight V-scan contributions;
    tokens at boundary patches can weight H-scans more. This is a ~4x wider
    information bottleneck before the classification decision.

    WHY NOT CLS TOKEN:
    ──────────────────
    In attention-based ViT, CLS works because attention is global — CLS
    simultaneously attends to every patch. In Mamba, tokens are sequential.
    A prepended CLS sees nothing before it; appended CLS only has the last
    hidden state. GAP naturally aggregates all positions equally and is
    the correct pooling strategy for SSM architectures.
    """

    def __init__(self, dim, H_p, W_p, d_state, d_conv, expand,
                 drop_path=0.0, layer_scale_init=1.0):
        super().__init__()
        self.H_p = H_p
        self.W_p = W_p
        self.norm = nn.LayerNorm(dim)

        # 4 independent SSMs for 4 scan directions
        self.mamba_h_fwd = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)
        self.mamba_h_bwd = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)
        self.mamba_v_fwd = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)
        self.mamba_v_bwd = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)

        # FIX: Learned direction merge instead of blind elementwise sum.
        # Projects from D*4 back to D, learning how much each scan direction
        # contributes per token. Small init prevents early representation collapse.
        self.merge = nn.Linear(dim * 4, dim, bias=False)
        nn.init.trunc_normal_(self.merge.weight, std=0.02 / math.sqrt(2 * 4))

        # FIX: layer_scale_init=1.0 (was 0.1).
        # Init of 0.1 scales all SSM outputs by 0.1x from epoch 1, making
        # effective learning rate 10x smaller through these layers on a
        # small dataset with limited epochs. Fine for large-scale pretraining;
        # harmful here.
        self.layer_scale = LayerScale(dim, layer_scale_init)
        self.drop_path   = StochasticDepth(drop_path)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, D = x.shape
        H_p, W_p = self.H_p, self.W_p
        residual = x
        h = self.norm(x)                              # (B, N, D)
        h_2d = h.view(B, H_p, W_p, D)                # (B, H_p, W_p, D)

        # ── Direction 1: H-forward (raster: row by row, left→right) ──────
        y_hf = self.mamba_h_fwd(h_2d.reshape(B, N, D))              # (B, N, D)

        # ── Direction 2: H-backward (reverse raster) ──────────────────────
        y_hb = self.mamba_h_bwd(h_2d.reshape(B, N, D).flip([1])).flip([1])

        # ── Direction 3: V-forward (column-major: col by col, top→bottom) ─
        # Transpose so vertically adjacent freq patches are step=1 apart
        h_vf = h_2d.permute(0, 2, 1, 3).reshape(B, N, D)
        y_vf = self.mamba_v_fwd(h_vf)
        y_vf = y_vf.view(B, W_p, H_p, D).permute(0, 2, 1, 3).reshape(B, N, D)

        # ── Direction 4: V-backward (reverse column-major) ────────────────
        h_vb = h_2d.permute(0, 2, 1, 3).reshape(B, N, D).flip([1])
        y_vb = self.mamba_v_bwd(h_vb).flip([1])
        y_vb = y_vb.view(B, W_p, H_p, D).permute(0, 2, 1, 3).reshape(B, N, D)

        # ── FIX: Concat all 4 directions, project back to D ──────────────
        # Each token now has access to full horizontal AND vertical context
        # before the projection decides how to weight them.
        merged = torch.cat([y_hf, y_hb, y_vf, y_vb], dim=-1)  # (B, N, D*4)
        out    = self.layer_scale(self.merge(merged))           # (B, N, D)

        return residual + self.drop_path(out)


class VisionMamba(nn.Module):
    """
    Pure Vision Mamba — 4-directional 2D scan, GAP pooling.
    No CNN components. All fixes applied.
    """

    def __init__(self, cfg: Config):
        super().__init__()
        D = cfg.embed_dim
        self.patch_embed = PatchEmbed(cfg.img_size, cfg.patch_size,
                                      cfg.in_channels, D)
        H_p = self.patch_embed.H_p
        W_p = self.patch_embed.W_p
        N   = self.patch_embed.num_patches

        self.pos_embed = nn.Parameter(torch.zeros(1, N, D))
        self.pos_drop  = nn.Dropout(p=cfg.drop_rate)

        dpr = [x.item() for x in torch.linspace(0, cfg.drop_path_rate, cfg.depth)]

        self.layers = nn.ModuleList([
            VimBlock2D(
                dim=D, H_p=H_p, W_p=W_p,
                d_state=cfg.d_state, d_conv=cfg.d_conv, expand=cfg.expand,
                drop_path=dpr[i], layer_scale_init=cfg.layer_scale_init,
            )
            for i in range(cfg.depth)
        ])

        self.norm = nn.LayerNorm(D)

        # 2-layer MLP classification head
        hidden_dim = D * 2
        self.head = nn.Sequential(
            nn.LayerNorm(D),
            nn.Linear(D, hidden_dim),
            nn.GELU(),
            nn.Dropout(p=cfg.drop_rate),
            nn.Linear(hidden_dim, cfg.num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def get_features(self, x: torch.Tensor) -> torch.Tensor:
        """Return GAP embedding before head (for probing / contrastive losses)."""
        x = self.patch_embed(x)
        x = self.pos_drop(x + self.pos_embed)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return x.mean(dim=1)          # (B, D)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x)      # (B, 196, D)
        x = self.pos_drop(x + self.pos_embed)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)            # GAP over 196 patch tokens — correct for SSM
        return self.head(x)


# ── Smoke test ───────────────────────────────────────────────────────────────
_cfg = Config()
_m   = VisionMamba(_cfg).to(device)
_inp = torch.randn(2, 3, 224, 224, device=device)
_out = _m(_inp)
total_p = sum(p.numel() for p in _m.parameters())
print(f'VisionMamba output shape: {_out.shape}')
print(f'Parameters: {total_p:,} total')
print(f'Tokens per image: {_m.patch_embed.num_patches} '
      f'({_m.patch_embed.H_p}x{_m.patch_embed.W_p} grid, patch_size={_cfg.patch_size})')
print(f'SSMs per block: 4 (H-fwd, H-bwd, V-fwd, V-bwd)')
print(f'Total SSMs: {_cfg.depth * 4}')
print(f'Merge: concat({_cfg.embed_dim}*4) -> Linear -> {_cfg.embed_dim}  [learned direction weighting]')
del _m, _inp, _out
torch.cuda.empty_cache()


In [ ]:
# ============================================================
# LOSS + OPTIMIZER (AdamW only — SGD removed, was 36% worse)
# ============================================================

def build_criterion(cfg: Config) -> nn.Module:
    crit = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing).to(device)
    print(f'Criterion: CrossEntropyLoss(label_smoothing={cfg.label_smoothing})')
    return crit


def build_optimizer(model: nn.Module, cfg: Config):
    """AdamW with cosine schedule. SGD removed — degraded to ~32% vs AdamW 68%."""
    return torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )


def adjust_lr(optimizer, epoch: int, cfg: Config) -> float:
    """Linear warmup + cosine annealing."""
    base_lr = cfg.learning_rate
    if epoch < cfg.warmup_epochs:
        lr = base_lr * (epoch + 1) / cfg.warmup_epochs
    else:
        progress = (epoch - cfg.warmup_epochs) / max(cfg.epochs - cfg.warmup_epochs, 1)
        lr = base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    return lr


# ============================================================
# MIXUP (CutMix disabled — too destructive stacked w/ SpecAugment)
# ============================================================

def mixup_batch(images, labels, alpha=0.2, num_classes=NUM_CLASSES):
    if alpha <= 0.0:
        ya = F.one_hot(labels, num_classes).float()
        return images, ya, ya, 1.0
    lam  = np.random.beta(alpha, alpha)
    lam  = max(lam, 1.0 - lam)
    perm = torch.randperm(images.size(0), device=images.device)
    mixed = lam * images + (1.0 - lam) * images[perm]
    ya    = F.one_hot(labels,       num_classes).float()
    yb    = F.one_hot(labels[perm], num_classes).float()
    return mixed, ya, yb, lam


def augment_batch(images, labels, cfg):
    """Only Mixup (CutMix disabled via cutmix_alpha=0.0)."""
    return mixup_batch(images, labels, alpha=cfg.mixup_alpha)


# ============================================================
# METRICS
# ============================================================

def compute_metrics(true_labels: np.ndarray, preds: np.ndarray,
                    num_classes: int) -> Dict:
    acc = accuracy_score(true_labels, preds) * 100.0
    f1  = f1_score(true_labels, preds, average='macro', zero_division=0) * 100.0
    cm  = confusion_matrix(true_labels, preds, labels=list(range(num_classes)))
    fpr_list, fnr_list = [], []
    per_class_acc = {}
    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        fpr_list.append(fp / max(fp + tn, 1))
        fnr_list.append(fn / max(fn + tp, 1))
        total_i = cm[i, :].sum()
        per_class_acc[CLASS_NAMES[i]] = round(tp / total_i * 100.0, 1) if total_i > 0 else 0.0
    return {
        'accuracy':      round(acc, 2),
        'f1':            round(f1,  2),
        'fpr':           round(float(np.mean(fpr_list)) * 100.0, 2),
        'fnr':           round(float(np.mean(fnr_list)) * 100.0, 2),
        'per_class_acc': per_class_acc,
    }


# ============================================================
# TRAIN / EVAL LOOPS
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, cfg,
                    scaler: GradScaler) -> Tuple[float, float]:
    model.train()
    total_loss = total_correct = total_samples = nan_batches = 0

    for step, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        mixed, ya, yb, lam = augment_batch(images, labels, cfg)

        optimizer.zero_grad(set_to_none=True)
        with autocast(dtype=torch.float16):
            logits = model(mixed)
            log_p  = F.log_softmax(logits, dim=-1)
            loss   = lam * (-(ya * log_p).sum(-1).mean()) + \
                     (1.0 - lam) * (-(yb * log_p).sum(-1).mean())

        if torch.isnan(loss) or torch.isinf(loss):
            nan_batches += 1
            optimizer.zero_grad(set_to_none=True)
            if nan_batches <= 3:
                print(f'  WARNING: NaN/Inf loss at step {step}. Skipping.')
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        bs             = images.size(0)
        total_loss    += loss.item() * bs
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += bs

    if nan_batches > 0:
        print(f'  [nan_batches this epoch: {nan_batches}]')
    if total_samples == 0:
        return float('nan'), 0.0
    return total_loss / total_samples, total_correct / total_samples * 100.0


@torch.no_grad()
def evaluate(model, loader, criterion) -> Tuple[float, Dict]:
    model.eval()
    total_loss = total_samples = 0
    all_preds, all_labels_ = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast(dtype=torch.float16):
            logits = model(images)
            loss   = criterion(logits, labels)
        bs             = images.size(0)
        total_loss    += loss.item() * bs
        total_samples += bs
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels_.extend(labels.cpu().tolist())
    metrics = compute_metrics(np.array(all_labels_), np.array(all_preds), NUM_CLASSES)
    return total_loss / total_samples, metrics


print('Loss, optimizer (AdamW), Mixup, metrics, and training utilities defined.')
print(f'Augmentation: Mixup(alpha={config.mixup_alpha}) only — CutMix disabled.')
print(f'Criterion: CrossEntropyLoss(label_smoothing={config.label_smoothing})')


In [ ]:
def run_training(cfg: Config,
                 tr_paths, tr_labels, tr_snrs,
                 vl_paths, vl_labels, vl_snrs,
                 tag='vim', verbose=True) -> Dict:

    train_loader, val_loader = build_dataloaders(
        cfg, tr_paths, tr_labels, tr_snrs, vl_paths, vl_labels, vl_snrs)

    model = VisionMamba(cfg)
    if num_gpus > 1:
        model = nn.DataParallel(model)
        if verbose:
            print(f'  Using DataParallel on {num_gpus} GPUs')
    model = model.to(device)

    criterion = build_criterion(cfg)
    optimizer = build_optimizer(model, cfg)
    scaler    = GradScaler()

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    best_val_acc = 0.0
    best_metrics = best_state = None
    patience_ctr = 0

    if verbose:
        total_p = sum(p.numel() for p in model.parameters())
        print(f'\n{"="*70}')
        print(f'Training: {tag.upper()}  |  optimizer=AdamW  lr={cfg.learning_rate}')
        print(f'  ARCH: 4-dir 2D scan + concat-project merge  |  '
              f'{cfg.depth} blocks x 4 SSMs = {cfg.depth * 4} total Mambas')
        print(f'  patch={cfg.patch_size}  tokens={(cfg.img_size//cfg.patch_size)**2}  '
              f'embed={cfg.embed_dim}  depth={cfg.depth}  d_state={cfg.d_state}')
        print(f'  params={total_p:,}  layer_scale={cfg.layer_scale_init}  '
              f'drop_path={cfg.drop_path_rate}')
        print(f'  augment=Mixup({cfg.mixup_alpha})+FreqShift(14)+SpecAugment(1x16)')
        print(f'  loss=CE(ls={cfg.label_smoothing})  sampling=balanced  amp=float16')
        print(f'  epochs={cfg.epochs}  patience={cfg.patience}  warmup={cfg.warmup_epochs}')
        print(f'{"="*70}')

    for epoch in range(cfg.epochs):
        lr = adjust_lr(optimizer, epoch, cfg)
        history['lr'].append(lr)

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, cfg, scaler)
        val_loss, val_metrics = evaluate(model, val_loader, criterion)
        val_acc = val_metrics['accuracy']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_metrics = deepcopy(val_metrics)
            patience_ctr = 0
            raw_model  = model.module if isinstance(model, nn.DataParallel) else model
            best_state = deepcopy(raw_model.state_dict())
        else:
            patience_ctr += 1

        pca = val_metrics['per_class_acc']
        if verbose and (epoch % 5 == 0 or epoch == cfg.epochs - 1 or patience_ctr == 0):
            print(
                f'  Epoch {epoch+1:3d}/{cfg.epochs} | LR {lr:.6f} | '
                f'Train {train_acc:.1f}% | Val {val_acc:.1f}% | Best {best_val_acc:.1f}% | '
                f'[ar={pca["ardrone"]:.0f}% bg={pca["background"]:.0f}% '
                f'bp={pca["bepop"]:.0f}% ph={pca["phantom"]:.0f}%]'
            )

        if patience_ctr >= cfg.patience:
            if verbose:
                print(f'  Early stopping at epoch {epoch+1}  (patience={cfg.patience})')
            break

    if verbose and best_metrics:
        bpca = best_metrics['per_class_acc']
        print(f'\nBest Val Acc: {best_val_acc:.2f}%')
        print(f'  Acc={best_metrics["accuracy"]:.2f}%  F1={best_metrics["f1"]:.2f}%  '
              f'FPR={best_metrics["fpr"]:.2f}%  FNR={best_metrics["fnr"]:.2f}%')
        print('  Per-class: ' + '  '.join(f'{n}={v:.1f}%' for n, v in bpca.items()))

    del model, criterion, optimizer, scaler
    torch.cuda.empty_cache()

    return {
        'tag': tag, 'best_val_acc': best_val_acc,
        'best_metrics': best_metrics, 'best_state': best_state, 'history': history,
    }


print('run_training defined — AdamW only, all fixes applied.')


In [ ]:
os.makedirs('vim_dronerf_outputs', exist_ok=True)

result = run_training(
    config,
    train_paths, train_labels, train_snrs,
    val_paths,   val_labels,   val_snrs,
    tag='vim_dronerf_v3',
    verbose=True,
)

torch.save(result['best_state'], 'vim_dronerf_outputs/vim_best.pth')
print('\nWeights saved to vim_dronerf_outputs/vim_best.pth')

final_history = result['history']


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(final_history['train_loss']) + 1)

axes[0].plot(epochs_range, final_history['train_loss'], label='Train Loss')
axes[0].plot(epochs_range, final_history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss  (AdamW, cosine schedule)')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, final_history['train_acc'], label='Train Acc')
axes[1].plot(epochs_range, final_history['val_acc'],   label='Val Acc')
best_v = max(final_history['val_acc'])
axes[1].axhline(y=best_v, color='green', linestyle='--', alpha=0.6,
                label=f'Best val {best_v:.1f}%')
axes[1].axhline(y=25.0, color='grey', linestyle=':', alpha=0.5,
                label='random-chance 25%')
axes[1].set_title('Accuracy  (AdamW, cosine schedule)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Check train/val gap — healthy model should have train > val by 5-15%
# If gap is near 0 it means augmentation is still too strong (underfitting)
gap = [tr - vl for tr, vl in zip(final_history['train_acc'], final_history['val_acc'])]
avg_gap = np.mean(gap[-20:]) if len(gap) >= 20 else np.mean(gap)
print(f'Avg train-val gap (last 20 epochs): {avg_gap:.1f}%')
print(f'  (healthy range: 5-15%; ~0 = under-augmented; >20% = overfitting)')

plt.tight_layout()
plt.savefig('vim_dronerf_outputs/training_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')


In [ ]:
eval_model = VisionMamba(config).to(device)
eval_model.load_state_dict(result['best_state'])
eval_model.eval()

_, val_loader_eval = build_dataloaders(
    config, train_paths, train_labels, train_snrs,
    val_paths, val_labels, val_snrs)

crit_eval = build_criterion(config)

val_loss_final, val_metrics_final = evaluate(eval_model, val_loader_eval, crit_eval)

print('Final evaluation on best checkpoint:')
print(f'  Val Loss : {val_loss_final:.4f}')
print(f'  Accuracy : {val_metrics_final["accuracy"]:.2f}%')
print(f'  F1       : {val_metrics_final["f1"]:.2f}%')
print(f'  FPR      : {val_metrics_final["fpr"]:.2f}%')
print(f'  FNR      : {val_metrics_final["fnr"]:.2f}%')
print(f'\nPer-class accuracy:')
for name, acc_c in val_metrics_final['per_class_acc'].items():
    print(f'  {name:<12}: {acc_c:.1f}%')


In [ ]:
ALL_SNR_LEVELS = sorted(set(all_snrs.tolist()))
SNR_LABEL      = {100: 'clean'}
for s in ALL_SNR_LEVELS:
    if s != 100:
        SNR_LABEL[s] = f'{s} dB'

val_snrs_arr  = np.array(val_snrs)
snr_transform = get_val_transforms(config.img_size)

print(f'{"="*68}')
print(f'SNR ROBUSTNESS EVALUATION')
print(f'{"="*68}')
print(f'  {"SNR":<10} {"N":>6}   {"Accuracy":>10}   {"F1":>8}   {"FPR":>8}   {"FNR":>8}')
print(f'  {"-"*64}')

for snr_val in ALL_SNR_LEVELS:
    mask     = val_snrs_arr == snr_val
    s_paths  = [val_paths[i] for i in range(len(val_paths)) if mask[i]]
    s_labels = val_labels[mask]
    s_snrs   = val_snrs_arr[mask]
    if len(s_paths) == 0:
        continue
    s_ds = DroneRFDataset(s_paths, s_labels, s_snrs, transform=snr_transform)
    s_loader = DataLoader(s_ds, batch_size=config.batch_size,
                          shuffle=False, num_workers=2, pin_memory=True)
    _, s_metrics = evaluate(eval_model, s_loader, crit_eval)
    label = SNR_LABEL.get(snr_val, f'{snr_val} dB')
    print(f'  {label:<10} {len(s_paths):>6}   '
          f'Acc={s_metrics["accuracy"]:>6.1f}%   '
          f'F1={s_metrics["f1"]:>6.1f}%   '
          f'FPR={s_metrics["fpr"]:>5.1f}%   '
          f'FNR={s_metrics["fnr"]:>5.1f}%')


In [ ]:
all_preds_cm, all_true_cm = [], []
eval_model.eval()
with torch.no_grad():
    for images, labels in val_loader_eval:
        images = images.to(device, non_blocking=True)
        with autocast(dtype=torch.float16):
            logits = eval_model(images)
        all_preds_cm.extend(logits.argmax(1).cpu().tolist())
        all_true_cm.extend(labels.tolist())

val_preds   = np.array(all_preds_cm)
true_labels = np.array(all_true_cm)

cm       = confusion_matrix(true_labels, val_preds, labels=list(range(NUM_CLASSES)))
row_sums = cm.sum(axis=1, keepdims=True).clip(min=1)
cm_norm  = cm.astype(float) / row_sums

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, title in [
    (axes[0], cm,      'Raw Counts'),
    (axes[1], cm_norm, 'Row-Normalized'),
]:
    im = ax.imshow(data, interpolation='nearest',
                   cmap='Blues', vmin=0, vmax=(1 if data is cm_norm else None))
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(CLASS_NAMES, fontsize=9)
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    ax.set_title(f'VisionMamba v3  ({title})', fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            val = cm_norm[i, j] if data is cm_norm else cm[i, j]
            fmt = f'{val:.2f}' if data is cm_norm else str(int(val))
            ax.text(j, i, fmt, ha='center', va='center', fontsize=8,
                    color='white' if cm_norm[i, j] > 0.5 else 'black')

plt.tight_layout()
plt.savefig('vim_dronerf_outputs/confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print('Per-class accuracy:')
for i, name in enumerate(CLASS_NAMES):
    total   = cm[i, :].sum()
    correct = cm[i, i]
    acc_c   = correct / total * 100.0 if total > 0 else 0.0
    print(f'  {name:<12}: {acc_c:.1f}%  ({correct}/{total})')
print('Saved confusion_matrix.png')
